# Chunk Generation Pipeline (Google Colab)

Converted automatically from the provided Python script.

In [ ]:
#!/usr/bin/env python3
"""Production entry point for the Chunk Generation pipeline stage (Google Colab Migration).

Stage: ``06_Text_Chunking``. Executed via direct Python invocation on Google Colab.

Reads validated Final Documents, splits each document into semantically
coherent, retrieval-optimized chunks using an adaptive
paragraph -> sentence -> whitespace splitting strategy, assigns
deterministic chunk identity and sibling links, validates the resulting
schema, and writes the chunk dataset to Parquet on S3 alongside a run
manifest.

This module composes ``config.py``, ``logger.py``, ``schema.py``, and
``utils.py`` exactly as published; none of those modules are modified.
Every S3 location this stage reads from or writes to is sourced from
``config.settings.s3`` (``config.S3Config``); no S3 path is hardcoded
in this module.

Migration Note:
    This file fully replaces the Amazon EMR/Apache Spark implementation.
    It leverages PyArrow, Pandas, and Python's concurrent.futures (ThreadPool) to
    process data in configurable memory-safe batches, utilizing garbage
    collection and robust S3 checkpointing to guarantee execution
    resilience without YARN or Spark.
"""

from __future__ import annotations

import argparse
import concurrent.futures
import functools
import gc
import hashlib
import json
import os
import random
import re
import sys
import time
from dataclasses import dataclass, fields, replace
from datetime import datetime, timezone
from typing import Any, Callable, Dict, List, Optional, Sequence, Set, Tuple, TypeVar

import boto3
import botocore.exceptions
import pandas as pd
import psutil
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.dataset as ds
import pyarrow.parquet as pq
import s3fs
from tqdm import tqdm

from config import (
    ChunkConfig,
    MasterConfig,
    OutputConfig,
    S3Config,
    settings,
)
from logger import ExecutionTimer, get_logger
from schema import (
    CHUNK_OUTPUT_SCHEMA,
    FINAL_DOCUMENT_SCHEMA,
    ChunkOutputColumns,
    ExecutionMetadataColumns,
    InputColumns,
    PipelineMetadataColumns,
    SchemaValidationError,
    ValidationMetadataColumns,
)
from utils import (
    Stopwatch,
    count_characters,
    current_utc_timestamp,
    estimate_token_count,
    generate_chunk_id,
    generate_uuid4,
    has_minimum_alphanumeric_ratio,
    is_blank,
    normalize_paragraph_structure,
    normalize_text,
    normalize_whitespace,
    validate_positive_integer,
    validate_s3_uri,
)

_LOGGER = get_logger(__name__, settings.logging)
_S3_PATH_FIELD_SUFFIX: str = "_path"

F = TypeVar('F', bound=Callable[..., Any])

_TRANSIENT_ERRORS = (
    IOError, 
    OSError, 
    botocore.exceptions.BotoCoreError, 
    botocore.exceptions.ClientError, 
    TimeoutError, 
    ConnectionError
)

class ChunkGenerationPipelineError(Exception):
    """Raised for fatal, non-recoverable failures of the pipeline run."""


# ---------------------------------------------------------------------------
# Robust Execution & Resilience
# ---------------------------------------------------------------------------

def with_s3_retry(max_retries: int = 5, base_delay: float = 1.0, max_delay: float = 32.0) -> Callable[[F], F]:
    """Decorator providing exponential backoff strictly for transient S3 operations."""
    def decorator(func: F) -> F:
        @functools.wraps(func)
        def wrapper(*args: Any, **kwargs: Any) -> Any:
            delay = base_delay
            for attempt in range(1, max_retries + 1):
                try:
                    return func(*args, **kwargs)
                except _TRANSIENT_ERRORS as exc:
                    if attempt == max_retries:
                        _LOGGER.error("S3 operation %s failed after %d attempts: %s", func.__name__, max_retries, exc)
                        raise
                    jitter = random.uniform(0, 0.5 * delay)
                    sleep_time = delay + jitter
                    _LOGGER.warning(
                        "S3 operation %s encountered transient failure (attempt %d/%d). Retrying in %.2fs. Error: %s",
                        func.__name__, attempt, max_retries, sleep_time, exc
                    )
                    time.sleep(sleep_time)
                    delay = min(delay * 2, max_delay)
        return wrapper # type: ignore
    return decorator


def safe_release_arrow_memory() -> None:
    """Safely triggers PyArrow memory release if supported by the installed version."""
    try:
        if hasattr(pa, 'default_memory_pool'):
            pool = pa.default_memory_pool()
            if hasattr(pool, 'release_unused'):
                pool.release_unused()
    except Exception as exc:
        _LOGGER.debug("PyArrow memory release failed or unsupported: %s", exc)


# ---------------------------------------------------------------------------
# Adaptive chunking parameters
# ---------------------------------------------------------------------------

@dataclass(frozen=True)
class AdaptiveChunkingParameters:
    """Token-based sizing parameters for the adaptive chunking strategy."""
    target_min_tokens: int = 350
    target_max_tokens: int = 500
    min_tokens: int = 200
    max_tokens: int = 600
    overlap_tokens: int = 50
    chars_per_token: float = settings.chunk.chars_per_token
    paragraph_separator: str = settings.chunk.paragraph_separator
    min_alphanumeric_ratio: float = settings.chunk.min_alphanumeric_ratio

    def __post_init__(self) -> None:
        if self.chars_per_token <= 0:
            raise ValueError("chars_per_token must be strictly positive.")
        if not (0 < self.min_tokens <= self.target_min_tokens <= self.target_max_tokens <= self.max_tokens):
            raise ValueError("Token thresholds must satisfy strict ordering.")
        if self.overlap_tokens < 0 or self.overlap_tokens >= self.min_tokens:
            raise ValueError("overlap_tokens must be non-negative and smaller than min_tokens.")

    @property
    def min_chars(self) -> int:
        return max(1, round(self.min_tokens * self.chars_per_token))

    @property
    def max_chars(self) -> int:
        return max(self.min_chars, round(self.max_tokens * self.chars_per_token))

    @property
    def target_max_chars(self) -> int:
        return max(self.min_chars, round(self.target_max_tokens * self.chars_per_token))

    @property
    def overlap_chars(self) -> int:
        return max(0, round(self.overlap_tokens * self.chars_per_token))


# ---------------------------------------------------------------------------
# Configuration and Environment validation
# ---------------------------------------------------------------------------

class ConfigurationValidator:
    """Validates configuration, credentials, and connectivity."""

    @staticmethod
    def validate_chunk_config(chunk_config: ChunkConfig) -> None:
        if not (0 < chunk_config.min_chunk_chars <= chunk_config.target_chunk_chars <= chunk_config.max_chunk_chars):
            raise ChunkGenerationPipelineError("config.ChunkConfig has inconsistent character thresholds.")
        if chunk_config.chunk_overlap_chars < 0 or chunk_config.chunk_overlap_chars >= chunk_config.min_chunk_chars:
            raise ChunkGenerationPipelineError("config.ChunkConfig.chunk_overlap_chars must be valid.")
        validate_positive_integer(chunk_config.target_chunk_chars, "target_chunk_chars")

    @staticmethod
    def validate_s3_paths(s3_config: S3Config) -> None:
        invalid_paths = [
            f"{f.name}={getattr(s3_config, f.name)!r}" 
            for f in fields(s3_config) if f.name.endswith(_S3_PATH_FIELD_SUFFIX) 
            if not validate_s3_uri(getattr(s3_config, f.name))
        ]
        if invalid_paths:
            raise ChunkGenerationPipelineError(f"Invalid S3 URI(s): {', '.join(invalid_paths)}.")

    @staticmethod
    @with_s3_retry(max_retries=3)
    def validate_aws_environment(s3_config: S3Config) -> None:
        session = boto3.Session()
        if not session.get_credentials():
            raise ChunkGenerationPipelineError("AWS credentials not found.")
        s3_client = session.client('s3', region_name=s3_config.region_name)
        s3_client.head_bucket(Bucket=s3_config.bucket_name)

    @staticmethod
    def validate_all(
        chunk_config: ChunkConfig, chunking_parameters: AdaptiveChunkingParameters, s3_config: S3Config
    ) -> None:
        with ExecutionTimer(_LOGGER, "validate_configuration"):
            ConfigurationValidator.validate_chunk_config(chunk_config)
            ConfigurationValidator.validate_s3_paths(s3_config)
            ConfigurationValidator.validate_aws_environment(s3_config)


# ---------------------------------------------------------------------------
# Adaptive chunking algorithm
# ---------------------------------------------------------------------------

_SENTENCE_BOUNDARY_PATTERN = re.compile(r"(?<=[.!?])\s+(?=[A-Z0-9\"'(])")

class AdaptiveChunker:
    """Splits a single document's text into semantically coherent chunks."""

    def __init__(self, params: AdaptiveChunkingParameters) -> None:
        self._params = params

    def chunk_document(self, document_text: Optional[str]) -> List[Dict[str, Any]]:
        normalized = normalize_paragraph_structure(normalize_text(document_text))
        if is_blank(normalized):
            return []

        raw_paragraphs = [
            p.strip() for p in normalized.split(self._params.paragraph_separator) if not is_blank(p)
        ]
        if not raw_paragraphs:
            raw_paragraphs = [normalized]

        atomic_segments: List[str] = []
        for paragraph in raw_paragraphs:
            atomic_segments.extend(self._decompose_segment(paragraph))
        if not atomic_segments:
            return []

        packed_chunks = self._pack_segments(atomic_segments)
        merged_chunks = self._merge_tiny_chunks(packed_chunks)
        final_chunks = self._apply_overlap(merged_chunks)

        records: List[Dict[str, Any]] = []
        for chunk_text in final_chunks:
            cleaned = normalize_whitespace(chunk_text)
            if is_blank(cleaned) or not has_minimum_alphanumeric_ratio(cleaned, self._params.min_alphanumeric_ratio):
                continue
            records.append({
                "chunk_text": cleaned,
                "token_count": estimate_token_count(cleaned, self._params.chars_per_token),
                "character_count": count_characters(cleaned),
            })

        return [{"chunk_number": i, **record} for i, record in enumerate(records)]

    def _decompose_segment(self, segment: str) -> List[str]:
        segment = segment.strip()
        if not segment:
            return []
        if len(segment) <= self._params.max_chars:
            return [segment]

        sentences = [s.strip() for s in _SENTENCE_BOUNDARY_PATTERN.split(segment) if s.strip()]
        if len(sentences) > 1:
            decomposed: List[str] = []
            for sentence in sentences:
                decomposed.extend(self._decompose_segment(sentence))
            return decomposed

        return self._split_by_whitespace(segment)

    def _split_by_whitespace(self, text: str) -> List[str]:
        words, pieces, buffer = text.split(), [], ""
        for word in words:
            candidate = f"{buffer} {word}".strip() if buffer else word
            if len(candidate) <= self._params.max_chars:
                buffer = candidate
                continue
            if buffer:
                pieces.append(buffer)
                buffer = ""
            if len(word) <= self._params.max_chars:
                buffer = word
            else:
                pieces.append(word)
        if buffer:
            pieces.append(buffer)
        return pieces

    def _pack_segments(self, segments: Sequence[str]) -> List[str]:
        chunks, buffer = [], ""
        for segment in segments:
            candidate = f"{buffer} {segment}".strip() if buffer else segment
            if len(candidate) <= self._params.max_chars:
                buffer = candidate
                if len(buffer) >= self._params.target_max_chars:
                    chunks.append(buffer)
                    buffer = ""
                continue
            if buffer:
                chunks.append(buffer)
                buffer = ""
            buffer = segment
            if len(buffer) >= self._params.target_max_chars:
                chunks.append(buffer)
                buffer = ""
        if buffer:
            chunks.append(buffer)
        return chunks

    def _merge_tiny_chunks(self, chunks: Sequence[str]) -> List[str]:
        if not chunks:
            return []
        working_chunks, merged, index = list(chunks), [], 0
        while index < len(working_chunks):
            current = working_chunks[index]
            is_tiny = len(current) < self._params.min_chars
            if is_tiny and index + 1 < len(working_chunks):
                candidate = f"{current} {working_chunks[index + 1]}".strip()
                if len(candidate) <= self._params.max_chars:
                    working_chunks[index + 1] = candidate
                    index += 1
                    continue
            if is_tiny and merged:
                candidate = f"{merged[-1]} {current}".strip()
                if len(candidate) <= self._params.max_chars:
                    merged[-1] = candidate
                    index += 1
                    continue
            merged.append(current)
            index += 1
        return merged

    def _apply_overlap(self, chunks: Sequence[str]) -> List[str]:
        if len(chunks) <= 1 or self._params.overlap_chars <= 0:
            return list(chunks)
        overlapped = [chunks[0]]
        for index in range(1, len(chunks)):
            overlap_text = self._extract_overlap_tail(chunks[index - 1])
            current_chunk = chunks[index]
            overlapped.append(f"{overlap_text} {current_chunk}".strip() if overlap_text else current_chunk)
        return overlapped

    def _extract_overlap_tail(self, text: str) -> str:
        if self._params.overlap_chars <= 0 or not text:
            return ""
        tail = text[-self._params.overlap_chars:]
        first_space = tail.find(" ")
        if first_space > 0:
            tail = tail[first_space + 1:]
        return tail.strip()


# ---------------------------------------------------------------------------
# Checkpointing Mechanism
# ---------------------------------------------------------------------------

class CheckpointManager:
    """Manages idempotent S3-based checkpointing via atomic-like swaps."""

    def __init__(self, temporary_path: str, filesystem: s3fs.S3FileSystem, enabled: bool = True) -> None:
        self._fs = filesystem
        self._enabled = enabled
        self._checkpoint_dir = os.path.join(temporary_path, "checkpoints")
        self._checkpoint_file = os.path.join(self._checkpoint_dir, "chunk_generation_colab.json")
        self._processed_batches: Set[str] = set()
        
        if self._enabled:
            self._load_checkpoint()

    @with_s3_retry(max_retries=3)
    def _load_checkpoint(self) -> None:
        if self._fs.exists(self._checkpoint_file):
            try:
                with self._fs.open(self._checkpoint_file, 'r') as f:
                    data = json.load(f)
                    self._processed_batches = set(data.get("processed_batches", []))
                _LOGGER.info("Loaded checkpoint with %d completed batches.", len(self._processed_batches))
            except Exception as exc:
                _LOGGER.warning("Failed to parse checkpoint (corrupt). Starting fresh. %s", exc)
                self._processed_batches = set()
        else:
            self._fs.makedirs(self._checkpoint_dir, exist_ok=True)

    def is_processed(self, batch_id: str) -> bool:
        if not self._enabled:
            return False
        return batch_id in self._processed_batches

    def mark_processed(self, batch_id: str) -> None:
        if not self._enabled:
            return
        self._processed_batches.add(batch_id)
        self._save_checkpoint()

    @with_s3_retry(max_retries=5)
    def _save_checkpoint(self) -> None:
        if not self._enabled:
            return
        temp_file = f"{self._checkpoint_file}.tmp"
        with self._fs.open(temp_file, 'w') as f:
            json.dump({"processed_batches": list(self._processed_batches)}, f)
        self._fs.mv(temp_file, self._checkpoint_file)

    @with_s3_retry(max_retries=3)
    def clear_checkpoints(self) -> None:
        if self._fs.exists(self._checkpoint_file):
            self._fs.rm(self._checkpoint_file)
        self._processed_batches.clear()


# ---------------------------------------------------------------------------
# PyArrow Schema mapping
# ---------------------------------------------------------------------------

def get_pyarrow_output_schema() -> pa.Schema:
    """Provides a 100% compatible PyArrow schema based on Spark CHUNK_OUTPUT_SCHEMA."""
    return pa.schema([
        pa.field(ChunkOutputColumns.CHUNK_ID, pa.string(), nullable=False),
        pa.field(ChunkOutputColumns.PARENT_ASIN, pa.string(), nullable=False),
        pa.field(ChunkOutputColumns.CATEGORY, pa.string(), nullable=False),
        pa.field(ChunkOutputColumns.PRODUCT_TITLE, pa.string(), nullable=False),
        pa.field(ChunkOutputColumns.CHUNK_NUMBER, pa.int32(), nullable=False),
        pa.field(ChunkOutputColumns.TOTAL_CHUNKS, pa.int32(), nullable=False),
        pa.field(ChunkOutputColumns.CHUNK_TEXT, pa.string(), nullable=False),
        pa.field(ChunkOutputColumns.TOKEN_COUNT, pa.int32(), nullable=False),
        pa.field(ChunkOutputColumns.CHARACTER_COUNT, pa.int32(), nullable=False),
        pa.field(ChunkOutputColumns.CREATED_TIMESTAMP, pa.timestamp('us', tz='UTC'), nullable=False),
        pa.field(ChunkOutputColumns.PREVIOUS_CHUNK_ID, pa.string(), nullable=True),
        pa.field(ChunkOutputColumns.NEXT_CHUNK_ID, pa.string(), nullable=True),
    ])


# ---------------------------------------------------------------------------
# Document Processing execution
# ---------------------------------------------------------------------------

def process_document_row(
    row_dict: Dict[str, Any], chunker: AdaptiveChunker, timestamp: datetime
) -> Tuple[List[Dict[str, Any]], int, int]:
    """Worker function for ThreadPool chunk extraction."""
    parent_asin = row_dict[InputColumns.PARENT_ASIN]
    category = row_dict[InputColumns.CATEGORY]
    title = row_dict[InputColumns.PRODUCT_TITLE]
    text = row_dict[InputColumns.DOCUMENT_TEXT]

    chunks = chunker.chunk_document(text)
    total_chunks = len(chunks)
    output_rows = []
    
    total_tokens = 0
    total_chars = 0
    
    for chunk in chunks:
        chunk_num = chunk["chunk_number"]
        chunk_id = generate_chunk_id(parent_asin, chunk_num)
        prev_id = generate_chunk_id(parent_asin, chunk_num - 1) if chunk_num > 0 else None
        next_id = generate_chunk_id(parent_asin, chunk_num + 1) if chunk_num + 1 < total_chunks else None

        total_tokens += chunk["token_count"]
        total_chars += chunk["character_count"]

        output_rows.append({
            ChunkOutputColumns.CHUNK_ID: chunk_id,
            ChunkOutputColumns.PARENT_ASIN: parent_asin,
            ChunkOutputColumns.CATEGORY: category,
            ChunkOutputColumns.PRODUCT_TITLE: title,
            ChunkOutputColumns.CHUNK_NUMBER: chunk_num,
            ChunkOutputColumns.TOTAL_CHUNKS: total_chunks,
            ChunkOutputColumns.CHUNK_TEXT: chunk["chunk_text"],
            ChunkOutputColumns.TOKEN_COUNT: chunk["token_count"],
            ChunkOutputColumns.CHARACTER_COUNT: chunk["character_count"],
            ChunkOutputColumns.CREATED_TIMESTAMP: timestamp,
            ChunkOutputColumns.PREVIOUS_CHUNK_ID: prev_id,
            ChunkOutputColumns.NEXT_CHUNK_ID: next_id,
        })
    return output_rows, total_tokens, total_chars


class ChunkGenerator:
    """Pure Python parallel processor for generating chunks."""
    
    def __init__(self, params: AdaptiveChunkingParameters):
        self._params = params
        self._chunker = AdaptiveChunker(params)
        self._max_workers = min(32, max(1, (os.cpu_count() or 1) * 2))
    
    def generate_batch(self, df: pd.DataFrame) -> Tuple[pa.Table, Set[str], int, int]:
        """Processes a dataframe and returns the Table, processed ASINs, tokens, and chars."""
        df = df[df[InputColumns.PARENT_ASIN].fillna("").astype(str).str.strip().astype(bool) &
                df[InputColumns.CATEGORY].fillna("").astype(str).str.strip().astype(bool) &
                df[InputColumns.DOCUMENT_TEXT].fillna("").astype(str).str.strip().astype(bool)]
        
        valid_input_asins = set(df[InputColumns.PARENT_ASIN].unique())

        if df.empty:
            return pa.Table.from_batches([], schema=get_pyarrow_output_schema()), valid_input_asins, 0, 0

        row_dicts = df.to_dict(orient='records')
        timestamp = current_utc_timestamp()
        
        all_chunk_records = []
        batch_tokens = 0
        batch_chars = 0
        
        process_func = functools.partial(process_document_row, chunker=self._chunker, timestamp=timestamp)

        def process_chunk(chunk_rows: List[Dict[str, Any]]) -> Tuple[List[Dict[str, Any]], int, int]:
            chunk_out = []
            t_tokens = 0
            t_chars = 0
            for row in chunk_rows:
                out, t, c = process_func(row)
                chunk_out.extend(out)
                t_tokens += t
                t_chars += c
            return chunk_out, t_tokens, t_chars

        chunk_size = 100
        chunks = [row_dicts[i:i + chunk_size] for i in range(0, len(row_dicts), chunk_size)]

        with concurrent.futures.ThreadPoolExecutor(max_workers=self._max_workers) as executor:
            for output_rows, doc_tokens, doc_chars in executor.map(process_chunk, chunks):
                if output_rows:
                    all_chunk_records.extend(output_rows)
                    batch_tokens += doc_tokens
                    batch_chars += doc_chars
                    
        if not all_chunk_records:
            return pa.Table.from_batches([], schema=get_pyarrow_output_schema()), valid_input_asins, 0, 0
            
        output_df = pd.DataFrame(all_chunk_records)

        if len(output_df[ChunkOutputColumns.CHUNK_ID].unique()) != len(output_df):
            raise ChunkGenerationPipelineError("Duplicate chunk_ids generated within the current batch.")

        table = pa.Table.from_pandas(output_df, schema=get_pyarrow_output_schema(), preserve_index=False)
        return table, valid_input_asins, batch_tokens, batch_chars


# ---------------------------------------------------------------------------
# Metadata builder
# ---------------------------------------------------------------------------

class MetadataBuilder:
    def __init__(self, run_id: str, execution_mode: str, master_config: MasterConfig) -> None:
        self._run_id = run_id
        self._execution_mode = execution_mode
        self._config = master_config

    def build_pipeline_metadata(self, source_record_count: int) -> Dict[str, Any]:
        config_hash = hashlib.sha256(str(self._config).encode('utf-8')).hexdigest()
        return {
            PipelineMetadataColumns.PROJECT_NAME: self._config.project.project_name,
            PipelineMetadataColumns.PIPELINE_STAGE: self._config.project.pipeline_stage,
            PipelineMetadataColumns.PRODUCT_CATEGORIES: ", ".join(self._config.project.product_categories),
            PipelineMetadataColumns.SOURCE_RECORD_COUNT: source_record_count,
            "pipeline_version": "1.0.0",
            "schema_version": "1.0.0",
            "pyarrow_version": pa.__version__,
            "pandas_version": pd.__version__,
            "execution_mode": self._execution_mode,
            "configuration_hash": config_hash,
        }

    def build_execution_metadata(
        self,
        started_at: datetime,
        completed_at: datetime,
        elapsed_seconds: float,
        input_record_count: int,
        output_record_count: int,
    ) -> Dict[str, Any]:
        return {
            ExecutionMetadataColumns.RUN_ID: self._run_id,
            ExecutionMetadataColumns.ENVIRONMENT: self._config.runtime.environment,
            ExecutionMetadataColumns.EXECUTION_START_UTC: started_at.isoformat(),
            ExecutionMetadataColumns.EXECUTION_END_UTC: completed_at.isoformat(),
            ExecutionMetadataColumns.ELAPSED_SECONDS: int(round(elapsed_seconds)),
            ExecutionMetadataColumns.INPUT_RECORD_COUNT: input_record_count,
            ExecutionMetadataColumns.OUTPUT_RECORD_COUNT: output_record_count,
        }

    def build_validation_metadata(
        self, 
        total_input_documents: int,
        processed_documents: int,
        skipped_documents: int,
        total_chunks: int,
        duplicate_chunk_ids: int,
        orphan_chunk_ids: int,
        documents_with_no_chunks: int,
        schema_validation_passed: bool,
        execution_duration_seconds: float,
    ) -> Dict[str, Any]:
        failed_records = duplicate_chunk_ids + orphan_chunk_ids
        passed_records = max(0, total_chunks - failed_records)
        failure_rate = failed_records / max(1, total_chunks)
        passed = (failure_rate <= self._config.validation.max_allowed_failure_rate) and schema_validation_passed

        return {
            ValidationMetadataColumns.TOTAL_RECORDS: total_chunks,
            ValidationMetadataColumns.PASSED_RECORDS: passed_records,
            ValidationMetadataColumns.FAILED_RECORDS: failed_records,
            ValidationMetadataColumns.FAILURE_RATE: f"{failure_rate:.4f}",
            ValidationMetadataColumns.MAX_ALLOWED_FAILURE_RATE: str(self._config.validation.max_allowed_failure_rate),
            ValidationMetadataColumns.VALIDATION_PASSED: str(passed),
            ValidationMetadataColumns.VALIDATED_AT_UTC: current_utc_timestamp().isoformat(),
            "total_input_documents": total_input_documents,
            "processed_documents": processed_documents,
            "skipped_documents": skipped_documents,
            "generated_chunks": total_chunks,
            "average_chunks_per_document": total_chunks / max(1, processed_documents),
            "duplicate_chunk_ids": duplicate_chunk_ids,
            "orphan_chunk_ids": orphan_chunk_ids,
            "documents_with_no_chunks": documents_with_no_chunks,
            "schema_validation": str(schema_validation_passed),
            "validation_status": "PASSED" if passed else "FAILED",
            "execution_duration": float(execution_duration_seconds)
        }


# ---------------------------------------------------------------------------
# Output Writer
# ---------------------------------------------------------------------------

class OutputWriter:
    def __init__(self, filesystem: s3fs.S3FileSystem, execution_mode: str = "resume"):
        self._fs = filesystem
        self._execution_mode = execution_mode

    @with_s3_retry(max_retries=5)
    def write_batch(self, table: pa.Table, output_path: str, output_config: OutputConfig, batch_id: str) -> None:
        """Writes batch table to S3, safely avoiding duplicates."""
        actual_batch_id = f"{batch_id}_{generate_uuid4()[:8]}" if self._execution_mode == "append" else batch_id
        
        try:
            behavior = "overwrite_or_ignore" if self._execution_mode in ["resume", "overwrite", "append"] else "error"
            ds.write_dataset(
                table,
                base_dir=output_path,
                format="parquet",
                partitioning=[ChunkOutputColumns.CATEGORY],
                existing_data_behavior=behavior,
                basename_template=f"{actual_batch_id}_{{i}}.parquet",
                filesystem=self._fs
            )
        except (TypeError, ValueError):
            unique_output_path = f"{output_path.rstrip('/')}/{actual_batch_id}"
            
            if self._fs.exists(unique_output_path):
                if self._execution_mode == "resume":
                    _LOGGER.info("Batch %s already exists. Skipping write.", actual_batch_id)
                    return
                elif self._execution_mode == "overwrite":
                    self._fs.rm(unique_output_path, recursive=True)
            
            pq.write_to_dataset(
                table,
                root_path=unique_output_path,
                partition_cols=[ChunkOutputColumns.CATEGORY],
                compression=output_config.compression_codec,
                filesystem=self._fs,
                use_dictionary=True
            )

    @with_s3_retry(max_retries=3)
    def write_json_record(self, record: Dict[str, Any], path: str) -> None:
        with self._fs.open(path, 'w') as f:
            json.dump(record, f, indent=4)
        _LOGGER.info("Record written to '%s'.", path)


# ---------------------------------------------------------------------------
# Execution summary
# ---------------------------------------------------------------------------

@dataclass(frozen=True)
class ExecutionSummary:
    run_id: str
    environment: str
    input_record_count: int
    output_record_count: int
    elapsed_seconds: float
    documents_per_second: float
    chunks_per_second: float
    average_chunk_size: float
    average_token_count: float
    memory_usage_mb: float
    peak_memory_usage_mb: float

    def log(self) -> None:
        _LOGGER.info(
            "EXECUTION SUMMARY | run_id=%s | environment=%s | input_records=%d | "
            "output_records=%d | elapsed_seconds=%.2f | docs/sec=%.2f | chunks/sec=%.2f | "
            "avg_chunk_size=%.2f | avg_token_count=%.2f | peak_memory_mb=%.2f",
            self.run_id, self.environment, self.input_record_count,
            self.output_record_count, self.elapsed_seconds, self.documents_per_second, 
            self.chunks_per_second, self.average_chunk_size, self.average_token_count,
            self.peak_memory_usage_mb
        )


# ---------------------------------------------------------------------------
# Pipeline orchestrator
# ---------------------------------------------------------------------------

class ChunkGenerationPipeline:
    def __init__(
        self,
        chunking_parameters: AdaptiveChunkingParameters,
        execution_mode: str = "resume",
        master_config: MasterConfig = settings,
        s3_config: Optional[S3Config] = None,
    ) -> None:
        self._chunking_parameters = chunking_parameters
        self._execution_mode = execution_mode
        self._config = master_config
        self._s3_config = s3_config if s3_config is not None else master_config.s3
        self._fs = s3fs.S3FileSystem()

    def _get_process_memory_mb(self) -> float:
        """Returns the current Resident Set Size (RSS) memory of the process in MB."""
        return psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)

    def _calculate_adaptive_batch_size(self, dataset: ds.Dataset) -> int:
        """Determines a safe batch size based on Colab's available RAM and sampled row sizes."""
        available_ram_bytes = psutil.virtual_memory().available
        target_ram_bytes = available_ram_bytes * 0.15 
        
        try:
            sample_batch = next(dataset.to_batches(batch_size=1000))
            sample_df = sample_batch.to_pandas()
            est_doc_size_bytes = sample_df.memory_usage(deep=True).sum() / max(1, len(sample_df))
        except StopIteration:
            est_doc_size_bytes = 5000

        expansion_factor = 5
        calculated = int(target_ram_bytes / (max(1, est_doc_size_bytes) * expansion_factor))
        safe_batch = max(5000, min(100000, calculated))
        _LOGGER.info("Adaptive batch size calculated: %d documents per batch (est. size %.2f KB).", 
                     safe_batch, est_doc_size_bytes / 1024)
        return safe_batch

    def _run_post_write_validation(self, output_path: str) -> Tuple[int, int, int, bool]:
        """Perform a streaming read-after-write scan for schema and data integrity."""
        _LOGGER.info("Starting Streaming Post-Write Validation and Read-Back...")
        dataset = ds.dataset(output_path, format="parquet", filesystem=self._fs)
        
        schema_validation_passed = (dataset.schema == get_pyarrow_output_schema())
        if not schema_validation_passed:
            _LOGGER.error("Final dataset schema does not match the expected pyarrow schema.")
            
        seen_chunks = set()
        total_chunks = 0
        orphan_chunk_ids = 0
        
        for batch in dataset.to_batches(columns=[ChunkOutputColumns.CHUNK_ID, ChunkOutputColumns.PARENT_ASIN]):
            chunk_ids = batch[ChunkOutputColumns.CHUNK_ID].to_pylist()
            asins = batch[ChunkOutputColumns.PARENT_ASIN].to_pylist()
            
            total_chunks += len(chunk_ids)
            seen_chunks.update(chunk_ids)
            orphan_chunk_ids += sum(1 for asin in asins if not asin or str(asin).strip() == "")
            
            safe_release_arrow_memory()
                
        unique_chunks = len(seen_chunks)
        duplicate_chunk_ids = total_chunks - unique_chunks
        
        if duplicate_chunk_ids > 0:
            _LOGGER.error("Global Duplicate Validation Failed: %d total vs %d unique chunk_ids.", total_chunks, unique_chunks)
            
        _LOGGER.info("Post-Write Validation Passed: %d total unique chunks.", total_chunks)
        return total_chunks, duplicate_chunk_ids, orphan_chunk_ids, schema_validation_passed

    def _upload_execution_log(self, run_id: str) -> None:
        """Uploads the local execution log to S3 upon completion."""
        log_path = getattr(self._s3_config, 'manifest_output_path', None)
        if not log_path:
            return
            
        local_log_file = os.path.join(self._config.logging.log_file_directory, self._config.logging.log_file_name)
        if os.path.exists(local_log_file):
            destination = f"{log_path.rstrip('/')}/{run_id}/execution.log"
            try:
                self._fs.put(local_log_file, destination)
                _LOGGER.info("Execution log uploaded to %s", destination)
            except Exception as e:
                _LOGGER.warning("Failed to upload execution log to S3: %s", e)

    def run(self) -> ExecutionSummary:
        run_id = generate_uuid4()
        stopwatch = Stopwatch().start()
        started_at = current_utc_timestamp()
        peak_memory_mb = self._get_process_memory_mb()

        _LOGGER.info("APPLICATION START | run_id=%s | stage=%s | mode=%s", 
                     run_id, self._config.project.pipeline_stage, self._execution_mode)

        try:
            ConfigurationValidator.validate_all(self._config.chunk, self._chunking_parameters, self._s3_config)
            
            checkpoint_mgr = CheckpointManager(
                self._s3_config.temporary_path, 
                self._fs, 
                enabled=(self._execution_mode in ["resume", "overwrite"])
            )

            if self._execution_mode == "overwrite":
                _LOGGER.info("Execution mode OVERWRITE: Clearing output path and checkpoints.")
                if self._fs.exists(self._s3_config.chunk_output_path):
                    self._fs.rm(self._s3_config.chunk_output_path, recursive=True)
                checkpoint_mgr.clear_checkpoints()

            dataset = ds.dataset(self._s3_config.final_documents_input_path, format="parquet", filesystem=self._fs)
            generator = ChunkGenerator(self._chunking_parameters)
            writer = OutputWriter(self._fs, self._execution_mode)
            batch_size = self._calculate_adaptive_batch_size(dataset)

            total_input_records = dataset.count_rows()
            total_processed_input = 0
            total_skipped_documents = 0
            total_output_records = 0
            documents_with_no_chunks = 0
            
            global_total_tokens = 0
            global_total_chars = 0
            
            pbar = tqdm(
                total=total_input_records, 
                desc="Processing Documents", 
                unit="doc",
                bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]'
            )

            for fragment in dataset.get_fragments():
                fragment_name = os.path.basename(fragment.path)
                
                for batch in fragment.to_batches(batch_size=batch_size):
                    df_batch = batch.to_pandas()
                    input_count = len(df_batch)
                    if input_count == 0:
                        continue

                    min_asin = df_batch[InputColumns.PARENT_ASIN].fillna("").astype(str).min()
                    max_asin = df_batch[InputColumns.PARENT_ASIN].fillna("").astype(str).max()
                    batch_hash = hashlib.sha256(f"{fragment_name}_{min_asin}_{max_asin}".encode('utf-8')).hexdigest()
                    batch_id = f"batch_{batch_hash}"

                    if checkpoint_mgr.is_processed(batch_id):
                        _LOGGER.info("Skipping processed batch: %s", batch_id)
                        pbar.update(input_count)
                        
                        df_filtered = df_batch[df_batch[InputColumns.PARENT_ASIN].fillna("").astype(str).str.strip().astype(bool) &
                                               df_batch[InputColumns.CATEGORY].fillna("").astype(str).str.strip().astype(bool) &
                                               df_batch[InputColumns.DOCUMENT_TEXT].fillna("").astype(str).str.strip().astype(bool)]
                        total_processed_input += len(df_filtered)
                        total_skipped_documents += (input_count - len(df_filtered))
                        continue
                    
                    table_out, valid_input_asins, batch_tokens, batch_chars = generator.generate_batch(df_batch)
                    output_count = table_out.num_rows
                    processed_in_batch = len(valid_input_asins)
                    skipped_in_batch = input_count - processed_in_batch

                    if output_count > 0:
                        writer.write_batch(table_out, self._s3_config.chunk_output_path, self._config.output, batch_id)
                        asins_with_chunks = set(table_out.column(ChunkOutputColumns.PARENT_ASIN).to_pylist())
                        documents_with_no_chunks += (processed_in_batch - len(asins_with_chunks))
                    else:
                        documents_with_no_chunks += processed_in_batch
                    
                    total_processed_input += processed_in_batch
                    total_skipped_documents += skipped_in_batch
                    total_output_records += output_count
                    global_total_tokens += batch_tokens
                    global_total_chars += batch_chars
                    
                    checkpoint_mgr.mark_processed(batch_id)
                    
                    # Proactive Contextual Cleanup
                    del df_batch
                    del table_out
                    gc.collect()
                    safe_release_arrow_memory()
                    
                    current_mem = self._get_process_memory_mb()
                    if current_mem > peak_memory_mb:
                        peak_memory_mb = current_mem

                    pbar.set_postfix(chunks=total_output_records, mem=f"{current_mem:.0f}MB")
                    pbar.update(input_count)

            pbar.close()

            # Global Validation Integrity Check
            documents_with_chunks = total_processed_input - documents_with_no_chunks
            if total_processed_input != (documents_with_chunks + documents_with_no_chunks):
                raise ChunkGenerationPipelineError(
                    f"Validation Failed: Processed documents ({total_processed_input}) does not equal "
                    f"documents with chunks ({documents_with_chunks}) + documents with no chunks ({documents_with_no_chunks})."
                )

            total_chunks, duplicate_chunk_ids, orphan_chunk_ids, schema_validation_passed = self._run_post_write_validation(self._s3_config.chunk_output_path)

            completed_at = current_utc_timestamp()
            elapsed_seconds = stopwatch.stop()

            # Metadata and Manifest Generation
            metadata_builder = MetadataBuilder(run_id, self._execution_mode, self._config)
            pipeline_metadata = metadata_builder.build_pipeline_metadata(total_processed_input)
            execution_metadata = metadata_builder.build_execution_metadata(
                started_at, completed_at, elapsed_seconds, total_processed_input, total_chunks
            )
            validation_metadata = metadata_builder.build_validation_metadata(
                total_input_documents=total_input_records,
                processed_documents=total_processed_input,
                skipped_documents=total_skipped_documents,
                total_chunks=total_chunks,
                duplicate_chunk_ids=duplicate_chunk_ids,
                orphan_chunk_ids=orphan_chunk_ids,
                documents_with_no_chunks=documents_with_no_chunks,
                schema_validation_passed=schema_validation_passed,
                execution_duration_seconds=elapsed_seconds
            )

            writer.write_json_record(
                {**pipeline_metadata, **execution_metadata},
                f"{self._s3_config.manifest_output_path.rstrip('/')}/{run_id}/manifest.json"
            )
            writer.write_json_record(
                validation_metadata,
                f"{self._s3_config.chunk_validation_path.rstrip('/')}/{run_id}/validation_report.json"
            )

            summary = ExecutionSummary(
                run_id=run_id,
                environment=self._config.runtime.environment,
                input_record_count=total_processed_input,
                output_record_count=total_chunks,
                elapsed_seconds=elapsed_seconds,
                documents_per_second=total_processed_input / max(elapsed_seconds, 1.0),
                chunks_per_second=total_chunks / max(elapsed_seconds, 1.0),
                average_chunk_size=global_total_chars / max(total_chunks, 1),
                average_token_count=global_total_tokens / max(total_chunks, 1),
                memory_usage_mb=self._get_process_memory_mb(),
                peak_memory_usage_mb=peak_memory_mb,
            )
            summary.log()
            _LOGGER.info("APPLICATION COMPLETE | run_id=%s", run_id)
            return summary

        except Exception:
            _LOGGER.error("APPLICATION FAILED | run_id=%s", run_id, exc_info=True)
            raise
            
        finally:
            self._upload_execution_log(run_id)


# ---------------------------------------------------------------------------
# CLI entry point
# ---------------------------------------------------------------------------

def _parse_arguments(argv: Optional[List[str]] = None) -> argparse.Namespace:
    default_s3 = settings.s3
    parser = argparse.ArgumentParser(description="Hybrid RAG Chunk Generation stage (Colab Migration).")
    parser.add_argument("--input-path", default=default_s3.final_documents_input_path)
    parser.add_argument("--output-path", default=default_s3.chunk_output_path)
    parser.add_argument("--validation-path", default=default_s3.chunk_validation_path)
    parser.add_argument("--manifest-path", default=default_s3.manifest_output_path)
    parser.add_argument(
        "--execution-mode", 
        type=str, 
        choices=["overwrite", "resume", "append"], 
        default="resume",
        help="Execution mode for handling existing data and checkpoints."
    )
    return parser.parse_args(argv)

def _resolve_s3_config(arguments: argparse.Namespace) -> S3Config:
    return replace(
        settings.s3,
        final_documents_input_path=arguments.input_path,
        chunk_output_path=arguments.output_path,
        chunk_validation_path=arguments.validation_path,
        manifest_output_path=arguments.manifest_path,
    )

def main(argv: Optional[List[str]] = None) -> int:
    arguments = _parse_arguments(argv)
    try:
        s3_config = _resolve_s3_config(arguments)
    except ValueError as exc:
        _LOGGER.error("Invalid S3 configuration for this run: %s", exc)
        return 1

    pipeline = ChunkGenerationPipeline(
        chunking_parameters=AdaptiveChunkingParameters(), 
        execution_mode=arguments.execution_mode,
        s3_config=s3_config
    )

    try:
        pipeline.run()
    except Exception:
        return 1
    return 0

if __name__ == "__main__":
    sys.exit(main())